# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Authors (IDs): {[author['@id'] for author in metadata.author]}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

The dataset conforms to Croissant and references all entities (record sets, fields, columns) by their `@id`.

In [ ]:
# Get available record sets by @id
record_sets = []
if hasattr(metadata, 'recordSet') and isinstance(metadata.recordSet, list) and metadata.recordSet:
    record_sets = [rs['@id'] for rs in metadata.recordSet]
    print(f"Record sets found: {record_sets}")
else:
    # Fallback: try to get recordSet ids from Croissant directly
    # Use internal Croissant model if available
    croissant_entities = dataset._croissant['@graph'] if hasattr(dataset, '_croissant') and '@graph' in dataset._croissant else []
    for ent in croissant_entities:
        if ent.get('@type') == 'cr:RecordSet' or ent.get('@type') == 'RecordSet':
            record_sets.append(ent['@id'])
    print(f"Record sets found: {record_sets}")

# For each record set, print available fields/columns and their @id
croissant_entities = dataset._croissant['@graph'] if hasattr(dataset, '_croissant') and '@graph' in dataset._croissant else []
fields_by_recordset = {}
for rs_id in record_sets:
    print(f"\nRecordSet @id: {rs_id}")
    for ent in croissant_entities:
        if ent.get('@id') == rs_id:
            # Fields are under 'field' key (Croissant spec)
            fields = ent.get('field', [])
            field_ids = []
            for field in fields:
                if isinstance(field, dict):
                    field_ids.append(field.get('@id'))
                elif isinstance(field, str):
                    field_ids.append(field)
            fields_by_recordset[rs_id] = field_ids
            print(f"Fields (@id): {field_ids}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing all entities by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nDataFrame loaded for RecordSet {record_set_id}. Columns (@id):")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Choose the first available record set for further analysis
main_record_set_id = record_sets[0] if record_sets else None
main_df = dataframes.get(main_record_set_id, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and categorizing data.

We will:
- Select a numeric field (e.g., GAD-7 score field `@id`)
- Filter by a threshold
- Normalize the numeric field
- Group by a demographic field (e.g., education level `@id`)

**Note:** All columns are accessed via their full `@id`.

In [ ]:
# Example field IDs for the dataset (must reference by @id)
# You may need to replace these with the actual @ids from section 2 and section 3
# For demonstration, we'll get the first numeric and group fields from main_df columns
numeric_field_id = None
group_field_id = None

# Try to infer numeric and group field candidates from column names
for col in main_df.columns:
    # Assume GAD-7, PHQ-9, PSQ scores, or similar, may be numeric
    if any(x in col.lower() for x in ['gad', 'phq', 'psq', 'score', 'age']) and numeric_field_id is None:
        numeric_field_id = col
    # Assume education or marital status makes good group fields
    if any(x in col.lower() for x in ['education', 'marital', 'village']) and group_field_id is None:
        group_field_id = col
# If not found, pick first column as fallback
if numeric_field_id is None and len(main_df.columns) > 0:
    numeric_field_id = main_df.columns[0]
if group_field_id is None and len(main_df.columns) > 1:
    group_field_id = main_df.columns[1]

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected (@id): {group_field_id}")

# Filter records
threshold = 10
if numeric_field_id in main_df.columns:
    # Convert to numeric if not already
    filtered_df = main_df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Group by another field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields using their `@id`.

Examples:
- Histogram of numeric field
- Boxplot grouped by a demographic field

In [ ]:
# Visualize numeric field distribution
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot grouped by group field
if group_field_id in main_df.columns and numeric_field_id in main_df.columns:
    plt.figure(figsize=(10,6))
    main_df[group_field_id] = main_df[group_field_id].astype(str)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    main_df.dropna(subset=[numeric_field_id, group_field_id], inplace=True)
    main_df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated:
- How to load the FAIR² Kilifi dataset via Croissant schema using `mlcroissant`.
- Access to metadata, record sets, and fields using their `@id`.
- Extraction of data into pandas DataFrames.
- Basic EDA tasks: filtering, normalization, grouping.
- Visualization of survey-based mental health indicators.

**Key findings** (fill in once you examine results):
- Distribution patterns and group comparisons for the selected fields.
- Potential survey-based insights for Kilifi mental health.